# FREUID ConvNeXtV2 G4 96GB All-Train Run

G4 96GB Colab notebook for a higher-resolution ConvNeXtV2 Large final run. It trains on all labeled rows, without holding out a document type, then saves checkpoints/submission artifacts back to Drive.

In [ ]:
!pip -q install timm

import csv
import json
import random
import shutil
import subprocess
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import timm
import torch
from PIL import Image, ImageFile
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True

if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

device: cuda
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 1. Copy Drive Cache To Local Disk

In [ ]:
COMPETITION = 'the-freuid-challenge-2026-ijcai-ecai'

DRIVE_ROOT = Path('/content/drive/MyDrive/FREUID_2026')
DRIVE_ZIP_PATH = DRIVE_ROOT / f'{COMPETITION}.zip'
DRIVE_OUTPUT_ROOT = DRIVE_ROOT / 'outputs' / 'convnextv2_g4_672_alltrain'
RUN_NAME = datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_RUN_DIR = DRIVE_OUTPUT_ROOT / RUN_NAME
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)

WORK_DIR = Path('/content/freuid')
DATA_ROOT = WORK_DIR / 'data'
LOCAL_ZIP_PATH = WORK_DIR / DRIVE_ZIP_PATH.name
WORK_DIR.mkdir(parents=True, exist_ok=True)

def copy_with_progress(src: Path, dst: Path, chunk_mb: int = 256) -> None:
    src = Path(src)
    dst = Path(dst)
    total = src.stat().st_size
    copied = 0
    last_report_gb = -1
    dst.parent.mkdir(parents=True, exist_ok=True)
    with src.open('rb') as fsrc, dst.open('wb') as fdst:
        while True:
            block = fsrc.read(chunk_mb * 1024 * 1024)
            if not block:
                break
            fdst.write(block)
            copied += len(block)
            report_gb = int(copied / 1e9)
            if report_gb != last_report_gb:
                last_report_gb = report_gb
                print(f'copied {copied / 1e9:.1f}/{total / 1e9:.1f} GB', flush=True)

assert DRIVE_ZIP_PATH.exists(), f'Missing Drive zip: {DRIVE_ZIP_PATH}'

if not LOCAL_ZIP_PATH.exists() or LOCAL_ZIP_PATH.stat().st_size != DRIVE_ZIP_PATH.stat().st_size:
    print('Copying Drive zip to local /content...')
    copy_with_progress(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
else:
    print('Local zip already matches Drive cache.')

if not (DATA_ROOT / 'train_labels.csv').exists():
    print('Unzipping to local /content...')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', str(LOCAL_ZIP_PATH), '-d', str(DATA_ROOT)], check=True)
else:
    print('Local extracted data already exists.')

required = [
    DATA_ROOT / 'train_labels.csv',
    DATA_ROOT / 'sample_submission.csv',
    DATA_ROOT / 'train' / 'train',
    DATA_ROOT / 'public_test' / 'public_test',
]
for path in required:
    print(path, 'OK' if path.exists() else 'MISSING')
assert all(path.exists() for path in required)

print('Drive outputs:', DRIVE_RUN_DIR)
!df -h /content /content/drive

Local zip already matches Drive cache.
Local extracted data already exists.
/content/freuid/data/train_labels.csv OK
/content/freuid/data/sample_submission.csv OK
/content/freuid/data/train/train OK
/content/freuid/data/public_test/public_test OK
Drive outputs: /content/drive/MyDrive/FREUID_2026/outputs/convnextv2_g4_672_alltrain/20260702_102114
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   97G  140G  42% /
drive           236G  104G  133G  44% /content/drive


## 2. ConvNeXtV2 G4 Configuration

In [ ]:
MODEL_NAME = 'convnextv2_large.fcmae_ft_in22k_in1k_384'
IMAGE_SIZE = 672
BATCH_SIZE = 24       # G4 96GB target. Try 48 if reserved VRAM stays below ~85GB.
EPOCHS = 3            # Final all-train mode: no type-holdout early stopping.
LR = 1e-5
WEIGHT_DECAY = 0.05
NUM_WORKERS = 12
VAL_TYPE = None       # None means train on all labeled data.
LOG_EVERY = 50
DUMMY_SCORE = 0.0
MAX_GRAD_NORM = 1.0
PATIENCE = 2
MIN_DELTA = 1e-4

USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
SCALER = torch.cuda.amp.GradScaler(enabled=(USE_AMP and AMP_DTYPE == torch.float16))

config = {
    'model_name': MODEL_NAME,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'num_workers': NUM_WORKERS,
    'val_type': VAL_TYPE,
    'dummy_score': DUMMY_SCORE,
    'max_grad_norm': MAX_GRAD_NORM,
    'patience': PATIENCE,
    'min_delta': MIN_DELTA,
    'amp_dtype': str(AMP_DTYPE),
    'seed': SEED,
}
(DRIVE_RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
print(json.dumps(config, indent=2))

{
  "model_name": "convnextv2_large.fcmae_ft_in22k_in1k_384",
  "image_size": 672,
  "batch_size": 24,
  "epochs": 3,
  "lr": 1e-05,
  "weight_decay": 0.05,
  "num_workers": 12,
  "val_type": null,
  "dummy_score": 0.0,
  "max_grad_norm": 1.0,
  "patience": 2,
  "min_delta": 0.0001,
  "amp_dtype": "torch.bfloat16",
  "seed": 42
}


/tmp/ipykernel_4574/1210945579.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  SCALER = torch.cuda.amp.GradScaler(enabled=(USE_AMP and AMP_DTYPE == torch.float16))


## 3. Data And Helpers

In [ ]:
def resolve_image_path(data_root: Path, image_path: str | Path) -> Path:
    path = Path(image_path)
    candidates = [data_root / path]
    parts = path.parts
    if len(parts) >= 2 and parts[0] in {'train', 'public_test', 'train_sample'}:
        candidates.append(data_root / parts[0] / parts[0] / parts[-1])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not resolve {image_path}; tried {candidates}')

with (DATA_ROOT / 'train_labels.csv').open(newline='', encoding='utf-8') as f:
    all_rows = list(csv.DictReader(f))

for row in all_rows:
    row['resolved_path'] = str(resolve_image_path(DATA_ROOT, row['image_path']))

random.shuffle(all_rows)
if VAL_TYPE is None:
    train_rows = all_rows
    val_rows = []
    print('final all-train mode: using every labeled row for training')
else:
    train_rows = [row for row in all_rows if row['type'] != VAL_TYPE]
    val_rows = [row for row in all_rows if row['type'] == VAL_TYPE]

print('all:', len(all_rows), 'train:', len(train_rows), 'val:', len(val_rows), 'heldout:', VAL_TYPE)
print('train labels:', {label: sum(r['label'] == label for r in train_rows) for label in ['0', '1']})
if val_rows:
    print('val labels:', {label: sum(r['label'] == label for r in val_rows) for label in ['0', '1']})

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.88, 1.0), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.14, contrast=0.14, saturation=0.05)], p=0.5),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.12),
    transforms.RandomRotation(degrees=1.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

eval_tfms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

class FreuidDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row['resolved_path']) as image:
            image = image.convert('RGB')
            image = self.transform(image)
        return image, torch.tensor(float(row['label']), dtype=torch.float32)

class ImageIdDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = list(image_paths)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        path = self.image_paths[index]
        with Image.open(path) as image:
            image = image.convert('RGB')
            image = self.transform(image)
        return path.stem, image

loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
loader_kwargs = {k: v for k, v in loader_kwargs.items() if v is not None}

train_loader = DataLoader(
    FreuidDataset(train_rows, train_tfms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)
val_loader = None
if val_rows:
    val_loader = DataLoader(
        FreuidDataset(val_rows, eval_tfms),
        batch_size=BATCH_SIZE,
        shuffle=False,
        **loader_kwargs,
    )
print('train batches:', len(train_loader), 'val batches:', 0 if val_loader is None else len(val_loader))

final all-train mode: using every labeled row for training
all: 69352 train: 69352 val: 0 heldout: None
train labels: {'0': 40005, '1': 29347}
train batches: 2889 val batches: 0


## 4. Train And Save Checkpoints To Drive

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=1).to(device)
model = model.to(memory_format=torch.channels_last)

train_pos = sum(r['label'] == '1' for r in train_rows)
train_neg = sum(r['label'] == '0' for r in train_rows)
print('train neg/pos:', train_neg, train_pos)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

LOCAL_OUTPUT_DIR = Path('/content/freuid_outputs/convnextv2_g4_672')
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics_path = LOCAL_OUTPUT_DIR / 'metrics.jsonl'
best_loss = None
bad_epochs = 0

def cuda_mem_gb() -> str:
    if not torch.cuda.is_available():
        return 'cpu'
    allocated = torch.cuda.max_memory_allocated() / 1e9
    reserved = torch.cuda.max_memory_reserved() / 1e9
    return f'alloc={allocated:.1f}GB reserved={reserved:.1f}GB'

@torch.no_grad()
def evaluate() -> dict:
    if val_loader is None:
        return {'val_loss': None, 'val_accuracy': None}
    model.eval()
    loss_sum = 0.0
    total = 0
    correct = 0
    for images, labels in val_loader:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)
        probs = torch.sigmoid(logits.float())
        preds = (probs >= 0.5).float()
        loss_sum += loss.item() * labels.numel()
        correct += (preds == labels).sum().item()
        total += labels.numel()
    return {'val_loss': loss_sum / total, 'val_accuracy': correct / total}

for epoch in range(1, EPOCHS + 1):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    model.train()
    epoch_start = time.time()
    train_start = time.time()
    loss_sum = 0.0
    seen = 0

    for step, (images, labels) in enumerate(train_loader, 1):
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)

        if SCALER.is_enabled():
            SCALER.scale(loss).backward()
            SCALER.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            SCALER.step(optimizer)
            SCALER.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()

        batch_size = labels.numel()
        loss_sum += loss.item() * batch_size
        seen += batch_size

        if step % LOG_EVERY == 0 or step == len(train_loader):
            elapsed = max(time.time() - train_start, 1e-6)
            print(
                f'epoch {epoch}/{EPOCHS} step {step}/{len(train_loader)} '
                f'loss {loss.item():.4f} img/s {seen / elapsed:.1f} {cuda_mem_gb()}',
                flush=True,
            )

    val_metrics = evaluate()
    scheduler.step()

    metrics = {
        'epoch': epoch,
        'train_loss': loss_sum / seen,
        **val_metrics,
        'lr': scheduler.get_last_lr()[0],
        'elapsed_min': (time.time() - epoch_start) / 60,
        'best_loss_before_update': best_loss,
    }
    print(json.dumps(metrics), flush=True)

    with metrics_path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(metrics) + '\n')

    if val_loader is None:
        improved = True
    else:
        improved = best_loss is None or metrics['val_loss'] < best_loss - MIN_DELTA

    checkpoint = {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'epoch': epoch,
        'best_loss': metrics['val_loss'] if (val_loader is not None and improved) else best_loss,
        'model_name': MODEL_NAME,
        'image_size': IMAGE_SIZE,
        'config': config,
        'metrics': metrics,
    }
    last_path = LOCAL_OUTPUT_DIR / 'last.pt'
    torch.save(checkpoint, last_path)
    shutil.copy2(last_path, DRIVE_RUN_DIR / 'last.pt')
    shutil.copy2(metrics_path, DRIVE_RUN_DIR / 'metrics.jsonl')

    if improved:
        if val_loader is not None:
            best_loss = metrics['val_loss']
            print('new best:', best_loss, 'saved to Drive')
        bad_epochs = 0
        best_path = LOCAL_OUTPUT_DIR / 'best.pt'
        torch.save(checkpoint, best_path)
        shutil.copy2(best_path, DRIVE_RUN_DIR / 'best.pt')
        if val_loader is None:
            print('all-train checkpoint updated:', best_path, 'saved to Drive')
    else:
        bad_epochs += 1
        print(f'no val improvement: {bad_epochs}/{PATIENCE}', flush=True)
        if bad_epochs >= PATIENCE:
            print('early stopping: validation loss did not improve', flush=True)
            break

print('Drive checkpoint dir:', DRIVE_RUN_DIR)

train neg/pos: 40005 29347
epoch 1/3 step 50/2889 loss 0.3829 img/s 18.1 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 100/2889 loss 0.2949 img/s 21.2 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 150/2889 loss 0.1337 img/s 22.4 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 200/2889 loss 0.0253 img/s 23.1 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 250/2889 loss 0.0003 img/s 23.5 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 300/2889 loss 0.0005 img/s 23.8 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 350/2889 loss 0.0002 img/s 24.0 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 400/2889 loss 0.0001 img/s 24.2 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 450/2889 loss 0.0001 img/s 24.3 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 500/2889 loss 0.0001 img/s 24.4 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 550/2889 loss 0.0000 img/s 24.5 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 600/2889 loss 0.0000 img/s 24.6 alloc=78.5GB reserved=82.1GB
epoch 1/3 step 650/2889 loss 0.0000 img/s 24.7 alloc=78.5GB re

## 5. Infer Public Test And Build Full Submission

In [ ]:
best_path = LOCAL_OUTPUT_DIR / 'best.pt'
if not best_path.exists():
    best_path = DRIVE_RUN_DIR / 'best.pt'
checkpoint = torch.load(best_path, map_location='cpu')

model = timm.create_model(checkpoint['model_name'], pretrained=False, num_classes=1)
model.load_state_dict(checkpoint['model'])
model.to(device)
model = model.to(memory_format=torch.channels_last)
model.eval()

public_paths = sorted((DATA_ROOT / 'public_test' / 'public_test').glob('*.jpeg'))
public_loader = DataLoader(
    ImageIdDataset(public_paths, eval_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

predictions_path = LOCAL_OUTPUT_DIR / 'public_predictions_convnextv2_alltrain.csv'
with predictions_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'label'])
    writer.writeheader()
    with torch.no_grad():
        for step, (ids, images) in enumerate(public_loader, 1):
            images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(images).squeeze(1)
            probs = torch.sigmoid(logits.float()).cpu().tolist()
            for row_id, score in zip(ids, probs):
                writer.writerow({'id': row_id, 'label': f'{float(score):.8g}'})
            if step % 20 == 0 or step == len(public_loader):
                print(f'infer step {step}/{len(public_loader)}', flush=True)

def read_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

sample_rows = read_csv_rows(DATA_ROOT / 'sample_submission.csv')
prediction_rows = read_csv_rows(predictions_path)
predictions = {row['id']: row['label'] for row in prediction_rows}

submission_path = LOCAL_OUTPUT_DIR / 'submission_convnextv2_g4_672_alltrain.csv'
dummy_label = f'{DUMMY_SCORE:.8g}'
predicted_rows = 0
dummy_rows = 0
with submission_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'label'])
    writer.writeheader()
    for row in sample_rows:
        row_id = row['id']
        if row_id in predictions:
            label = predictions[row_id]
            predicted_rows += 1
        else:
            label = dummy_label
            dummy_rows += 1
        writer.writerow({'id': row_id, 'label': label})

submission_rows = read_csv_rows(submission_path)
assert len(submission_rows) == len(sample_rows) == 142818
assert [r['id'] for r in submission_rows] == [r['id'] for r in sample_rows]
assert len({r['id'] for r in submission_rows}) == 142818

summary = {
    'submission': str(submission_path),
    'total_rows': len(submission_rows),
    'predicted_rows': predicted_rows,
    'dummy_rows': dummy_rows,
    'dummy_score': DUMMY_SCORE,
    'checkpoint': str(best_path),
}
(LOCAL_OUTPUT_DIR / 'submission_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

for path in [predictions_path, submission_path, LOCAL_OUTPUT_DIR / 'submission_summary.json']:
    shutil.copy2(path, DRIVE_RUN_DIR / path.name)

print(json.dumps(summary, indent=2))
print('Saved artifacts to Drive:', DRIVE_RUN_DIR)

infer step 20/326
infer step 40/326
infer step 60/326
infer step 80/326
infer step 100/326
infer step 120/326
infer step 140/326
infer step 160/326
infer step 180/326
infer step 200/326
infer step 220/326
infer step 240/326
infer step 260/326
infer step 280/326
infer step 300/326
infer step 320/326
infer step 326/326
{
  "submission": "/content/freuid_outputs/convnextv2_g4_672/submission_convnextv2_g4_672_alltrain.csv",
  "total_rows": 142818,
  "predicted_rows": 7821,
  "dummy_rows": 134997,
  "dummy_score": 0.0,
  "checkpoint": "/content/freuid_outputs/convnextv2_g4_672/best.pt"
}
Saved artifacts to Drive: /content/drive/MyDrive/FREUID_2026/outputs/convnextv2_g4_672_alltrain/20260702_102114
